In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv('../1/1_data.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304
...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250


# Khởi tạo is_promotion

In [3]:
df['is_promotion'] = (df['dis_1'] > 0) | (df['dis_2'] > 0)
df


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs,is_promotion
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388,False
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464,False
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301,False
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833,False
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304,False
...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009,True
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398,True
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062,True
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250,False


In [4]:
targets = ['total_revenue', 'total_cogs']
features = df.columns.difference(targets)
features



Index(['category', 'city', 'dis_1', 'dis_2', 'is_promotion', 'order_date',
       'order_source', 'segment'],
      dtype='object')

# 1. Các đặc trưng về ngày, tháng

In [5]:
# đảm bảo đúng kiểu datetime
df['order_date'] = pd.to_datetime(df['order_date'])

# tạo các đặc trưng thời gian
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['dow'] = df['order_date'].dt.dayofweek      # 0=Monday, 6=Sunday
df['doy'] = df['order_date'].dt.dayofyear

df


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs,is_promotion,year,month,dow,doy
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388,False,2012,7,2,186
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464,False,2012,7,2,186
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301,False,2012,7,2,186
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833,False,2012,7,2,186
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304,False,2012,7,2,186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009,True,2022,12,5,365
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398,True,2022,12,5,365
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062,True,2022,12,5,365
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250,False,2022,12,5,365


In [6]:
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)


# 2. Global features

## a. Lagging Features

In [7]:
temp_rev_cogs = df.groupby('order_date')[['total_revenue', 'total_cogs']].sum().reset_index()
temp = df.merge(temp_rev_cogs, on='order_date', suffixes=('', '_daily_global'))


In [8]:
for i in ['total_revenue_daily_global', 'total_cogs_daily_global']:
    for j in [1, 3, 7, 14, 30, 60, 90, 180, 365]:
        temp[f'lag_{i}_{j}'] = temp[i].shift(j)


In [9]:
temp


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs,is_promotion,...,lag_total_revenue_daily_global_365,lag_total_cogs_daily_global_1,lag_total_cogs_daily_global_3,lag_total_cogs_daily_global_7,lag_total_cogs_daily_global_14,lag_total_cogs_daily_global_30,lag_total_cogs_daily_global_60,lag_total_cogs_daily_global_90,lag_total_cogs_daily_global_180,lag_total_cogs_daily_global_365
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464,False,...,NaN,3.982991e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301,False,...,NaN,3.982991e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833,False,...,NaN,3.982991e+06,3.982991e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304,False,...,NaN,3.982991e+06,3.982991e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009,True,...,3448729.2,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,3.022292e+06,3.170787e+06,3.513621e+06
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398,True,...,3448729.2,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,3.022292e+06,3.170787e+06,3.513621e+06
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062,True,...,3448729.2,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,3.022292e+06,3.170787e+06,3.513621e+06
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250,False,...,3448729.2,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,2.279288e+06,3.022292e+06,3.170787e+06,3.513621e+06


In [10]:
df_temp_1 = temp.copy()
del temp, temp_rev_cogs


## b. Rolling Window Features

In [11]:
temp = df_temp_1.copy()


In [12]:
for i in ['total_revenue_daily_global', 'total_cogs_daily_global']:
    for j in [1, 3, 7, 14, 30, 60, 90, 180, 365, 365 * 2, 365 * 3]:
        temp[f'rolling_mean_{i}_{j}'] = temp[i].shift(1).rolling(window=j).mean()
        temp[f'rolling_min_{i}_{j}'] = temp[i].shift(1).rolling(window=j).min()
        temp[f'rolling_max_{i}_{j}'] = temp[i].shift(1).rolling(window=j).max()
        if j != 1:  # Chỉ tính rolling std cho các cửa sổ nhỏ hơn hoặc bằng 14 ngày
            temp[f'rolling_std_{i}_{j}'] = temp[i].shift(1).rolling(window=j).std()
temp


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs,is_promotion,...,rolling_max_total_cogs_daily_global_365,rolling_std_total_cogs_daily_global_365,rolling_mean_total_cogs_daily_global_730,rolling_min_total_cogs_daily_global_730,rolling_max_total_cogs_daily_global_730,rolling_std_total_cogs_daily_global_730,rolling_mean_total_cogs_daily_global_1095,rolling_min_total_cogs_daily_global_1095,rolling_max_total_cogs_daily_global_1095,rolling_std_total_cogs_daily_global_1095
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009,True,...,3.513621e+06,390168.435923,2.665676e+06,1.160101e+06,3.513621e+06,729943.425088,2.349250e+06,1.160101e+06,3.513621e+06,756925.195619
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398,True,...,3.513621e+06,391376.622993,2.666878e+06,1.160101e+06,3.513621e+06,728580.178156,2.349457e+06,1.160101e+06,3.513621e+06,756875.158443
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062,True,...,3.513621e+06,392551.880109,2.668080e+06,1.160101e+06,3.513621e+06,727212.385417,2.349664e+06,1.160101e+06,3.513621e+06,756825.061521
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250,False,...,3.513621e+06,393694.502176,2.669282e+06,1.160101e+06,3.513621e+06,725840.021174,2.349870e+06,1.160101e+06,3.513621e+06,756774.904840


In [13]:
df_temp_2 = temp.copy()
del temp


In [14]:
df_temp_2


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs,is_promotion,...,rolling_max_total_cogs_daily_global_365,rolling_std_total_cogs_daily_global_365,rolling_mean_total_cogs_daily_global_730,rolling_min_total_cogs_daily_global_730,rolling_max_total_cogs_daily_global_730,rolling_std_total_cogs_daily_global_730,rolling_mean_total_cogs_daily_global_1095,rolling_min_total_cogs_daily_global_1095,rolling_max_total_cogs_daily_global_1095,rolling_std_total_cogs_daily_global_1095
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009,True,...,3.513621e+06,390168.435923,2.665676e+06,1.160101e+06,3.513621e+06,729943.425088,2.349250e+06,1.160101e+06,3.513621e+06,756925.195619
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398,True,...,3.513621e+06,391376.622993,2.666878e+06,1.160101e+06,3.513621e+06,728580.178156,2.349457e+06,1.160101e+06,3.513621e+06,756875.158443
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062,True,...,3.513621e+06,392551.880109,2.668080e+06,1.160101e+06,3.513621e+06,727212.385417,2.349664e+06,1.160101e+06,3.513621e+06,756825.061521
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250,False,...,3.513621e+06,393694.502176,2.669282e+06,1.160101e+06,3.513621e+06,725840.021174,2.349870e+06,1.160101e+06,3.513621e+06,756774.904840


# 3. Next 548 days Labels Engineering

In [15]:
temp = df_temp_2.copy()
temp


,order_date,segment,category,order_source,dis_1,dis_2,city,total_revenue,total_cogs,is_promotion,...,rolling_max_total_cogs_daily_global_365,rolling_std_total_cogs_daily_global_365,rolling_mean_total_cogs_daily_global_730,rolling_min_total_cogs_daily_global_730,rolling_max_total_cogs_daily_global_730,rolling_std_total_cogs_daily_global_730,rolling_mean_total_cogs_daily_global_1095,rolling_min_total_cogs_daily_global_1095,rolling_max_total_cogs_daily_global_1095,rolling_std_total_cogs_daily_global_1095
0,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Can Tho,13254.31,10205.253388,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Da Nang,1641.85,1337.836464,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-07-04,Activewear,Outdoor,direct,0.0,0.0,Hai Phong,8995.77,8130.592301,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Bac Giang,3326.75,3288.465833,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-07-04,Activewear,Outdoor,email_campaign,0.0,0.0,Can Tho,12589.92,9113.723304,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Bien Hoa,13093.00,15944.774009,True,...,3.513621e+06,390168.435923,2.665676e+06,1.160101e+06,3.513621e+06,729943.425088,2.349250e+06,1.160101e+06,3.513621e+06,756925.195619
572687,2022-12-31,Premium,Outdoor,social_media,20.0,0.0,Da Nang,12423.60,13178.149398,True,...,3.513621e+06,391376.622993,2.666878e+06,1.160101e+06,3.513621e+06,728580.178156,2.349457e+06,1.160101e+06,3.513621e+06,756875.158443
572688,2022-12-31,Standard,Streetwear,social_media,20.0,0.0,Vinh Long,12797.73,12043.796062,True,...,3.513621e+06,392551.880109,2.668080e+06,1.160101e+06,3.513621e+06,727212.385417,2.349664e+06,1.160101e+06,3.513621e+06,756825.061521
572689,2022-12-31,Trendy,GenZ,email_campaign,0.0,0.0,Ninh Binh,13738.50,12917.126250,False,...,3.513621e+06,393694.502176,2.669282e+06,1.160101e+06,3.513621e+06,725840.021174,2.349870e+06,1.160101e+06,3.513621e+06,756774.904840


In [16]:
# Drop 2 cột 'total_revenue' và 'total_cogs' vì chúng ta chỉ quan tâm tới 'total_revenue_daily_global' và 'total_cogs_daily_global'
temp.drop(columns=['total_revenue', 'total_cogs','segment','category','order_source','dis_1','dis_2','city'], inplace=True)
temp


,order_date,is_promotion,year,month,dow,doy,dow_sin,dow_cos,month_sin,month_cos,...,rolling_max_total_cogs_daily_global_365,rolling_std_total_cogs_daily_global_365,rolling_mean_total_cogs_daily_global_730,rolling_min_total_cogs_daily_global_730,rolling_max_total_cogs_daily_global_730,rolling_std_total_cogs_daily_global_730,rolling_mean_total_cogs_daily_global_1095,rolling_min_total_cogs_daily_global_1095,rolling_max_total_cogs_daily_global_1095,rolling_std_total_cogs_daily_global_1095
0,2012-07-04,False,2012,7,2,186,0.974928,-0.222521,-5.000000e-01,-0.866025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012-07-04,False,2012,7,2,186,0.974928,-0.222521,-5.000000e-01,-0.866025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2012-07-04,False,2012,7,2,186,0.974928,-0.222521,-5.000000e-01,-0.866025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2012-07-04,False,2012,7,2,186,0.974928,-0.222521,-5.000000e-01,-0.866025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2012-07-04,False,2012,7,2,186,0.974928,-0.222521,-5.000000e-01,-0.866025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,390168.435923,2.665676e+06,1.160101e+06,3.513621e+06,729943.425088,2.349250e+06,1.160101e+06,3.513621e+06,756925.195619
572687,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,391376.622993,2.666878e+06,1.160101e+06,3.513621e+06,728580.178156,2.349457e+06,1.160101e+06,3.513621e+06,756875.158443
572688,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,392551.880109,2.668080e+06,1.160101e+06,3.513621e+06,727212.385417,2.349664e+06,1.160101e+06,3.513621e+06,756825.061521
572689,2022-12-31,False,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,393694.502176,2.669282e+06,1.160101e+06,3.513621e+06,725840.021174,2.349870e+06,1.160101e+06,3.513621e+06,756774.904840


In [17]:
temp.dropna(inplace=True)
temp


,order_date,is_promotion,year,month,dow,doy,dow_sin,dow_cos,month_sin,month_cos,...,rolling_max_total_cogs_daily_global_365,rolling_std_total_cogs_daily_global_365,rolling_mean_total_cogs_daily_global_730,rolling_min_total_cogs_daily_global_730,rolling_max_total_cogs_daily_global_730,rolling_std_total_cogs_daily_global_730,rolling_mean_total_cogs_daily_global_1095,rolling_min_total_cogs_daily_global_1095,rolling_max_total_cogs_daily_global_1095,rolling_std_total_cogs_daily_global_1095
1095,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,113897.890674,3.672129e+06,1.808623e+06,4.458811e+06,931663.658591,3.441569e+06,1.808623e+06,4.458811e+06,960735.517164
1096,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,112831.002902,3.675349e+06,1.808623e+06,4.458811e+06,930312.212089,3.442004e+06,1.808623e+06,4.458811e+06,961088.166726
1097,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,111742.365234,3.678569e+06,1.808623e+06,4.458811e+06,928947.623155,3.442438e+06,1.808623e+06,4.458811e+06,961440.490362
1098,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,110631.335601,3.681789e+06,1.808623e+06,4.458811e+06,927569.833784,3.442873e+06,1.808623e+06,4.458811e+06,961792.488431
1099,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,109497.232394,3.685009e+06,1.808623e+06,4.458811e+06,926178.785066,3.443307e+06,1.808623e+06,4.458811e+06,962144.161289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,390168.435923,2.665676e+06,1.160101e+06,3.513621e+06,729943.425088,2.349250e+06,1.160101e+06,3.513621e+06,756925.195619
572687,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,391376.622993,2.666878e+06,1.160101e+06,3.513621e+06,728580.178156,2.349457e+06,1.160101e+06,3.513621e+06,756875.158443
572688,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,392551.880109,2.668080e+06,1.160101e+06,3.513621e+06,727212.385417,2.349664e+06,1.160101e+06,3.513621e+06,756825.061521
572689,2022-12-31,False,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,393694.502176,2.669282e+06,1.160101e+06,3.513621e+06,725840.021174,2.349870e+06,1.160101e+06,3.513621e+06,756774.904840


In [18]:
temp.drop_duplicates(inplace=True)
temp


,order_date,is_promotion,year,month,dow,doy,dow_sin,dow_cos,month_sin,month_cos,...,rolling_max_total_cogs_daily_global_365,rolling_std_total_cogs_daily_global_365,rolling_mean_total_cogs_daily_global_730,rolling_min_total_cogs_daily_global_730,rolling_max_total_cogs_daily_global_730,rolling_std_total_cogs_daily_global_730,rolling_mean_total_cogs_daily_global_1095,rolling_min_total_cogs_daily_global_1095,rolling_max_total_cogs_daily_global_1095,rolling_std_total_cogs_daily_global_1095
1095,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,113897.890674,3.672129e+06,1.808623e+06,4.458811e+06,931663.658591,3.441569e+06,1.808623e+06,4.458811e+06,960735.517164
1096,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,112831.002902,3.675349e+06,1.808623e+06,4.458811e+06,930312.212089,3.442004e+06,1.808623e+06,4.458811e+06,961088.166726
1097,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,111742.365234,3.678569e+06,1.808623e+06,4.458811e+06,928947.623155,3.442438e+06,1.808623e+06,4.458811e+06,961440.490362
1098,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,110631.335601,3.681789e+06,1.808623e+06,4.458811e+06,927569.833784,3.442873e+06,1.808623e+06,4.458811e+06,961792.488431
1099,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,4.458811e+06,109497.232394,3.685009e+06,1.808623e+06,4.458811e+06,926178.785066,3.443307e+06,1.808623e+06,4.458811e+06,962144.161289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,390168.435923,2.665676e+06,1.160101e+06,3.513621e+06,729943.425088,2.349250e+06,1.160101e+06,3.513621e+06,756925.195619
572687,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,391376.622993,2.666878e+06,1.160101e+06,3.513621e+06,728580.178156,2.349457e+06,1.160101e+06,3.513621e+06,756875.158443
572688,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,392551.880109,2.668080e+06,1.160101e+06,3.513621e+06,727212.385417,2.349664e+06,1.160101e+06,3.513621e+06,756825.061521
572689,2022-12-31,False,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,3.513621e+06,393694.502176,2.669282e+06,1.160101e+06,3.513621e+06,725840.021174,2.349870e+06,1.160101e+06,3.513621e+06,756774.904840


In [19]:
temp.iloc[0]


order_date                                   2012-07-12 00:00:00
is_promotion                                               False
year                                                        2012
month                                                          7
dow                                                            3
                                                    ...         
rolling_std_total_cogs_daily_global_730            931663.658591
rolling_mean_total_cogs_daily_global_1095         3441569.172461
rolling_min_total_cogs_daily_global_1095          1808622.794866
rolling_max_total_cogs_daily_global_1095          4458811.272265
rolling_std_total_cogs_daily_global_1095           960735.517164
Name: 1095, Length: 116, dtype: object

In [20]:
HORIZON = 548

df = temp.copy()

# tạo label cho revenue
for i in range(1, HORIZON + 1):
    df[f'y_rev_t+{i}'] = df['total_revenue_daily_global'].shift(-i)

# tạo label cho cogs
for i in range(1, HORIZON + 1):
    df[f'y_cogs_t+{i}'] = df['total_cogs_daily_global'].shift(-i)


/var/folders/36/khn_y8zn7qqflczfpst8cbfc0000gn/T/ipykernel_40359/348648247.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'y_rev_t+{i}'] = df['total_revenue_daily_global'].shift(-i)
/var/folders/36/khn_y8zn7qqflczfpst8cbfc0000gn/T/ipykernel_40359/348648247.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'y_rev_t+{i}'] = df['total_revenue_daily_global'].shift(-i)
/var/folders/36/khn_y8zn7qqflczfpst8cbfc0000gn/T/ipykernel_40359/348648247.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually th

In [21]:
df


,order_date,is_promotion,year,month,dow,doy,dow_sin,dow_cos,month_sin,month_cos,...,y_cogs_t+539,y_cogs_t+540,y_cogs_t+541,y_cogs_t+542,y_cogs_t+543,y_cogs_t+544,y_cogs_t+545,y_cogs_t+546,y_cogs_t+547,y_cogs_t+548
1095,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,3.154482e+06,3.154482e+06,3.154482e+06,3.154482e+06,3.154482e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06
1096,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,3.154482e+06,3.154482e+06,3.154482e+06,3.154482e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06
1097,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,3.154482e+06,3.154482e+06,3.154482e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06
1098,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,3.154482e+06,3.154482e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06
1099,2012-07-12,False,2012,7,3,194,0.433884,-0.900969,-5.000000e-01,-0.866025,...,3.154482e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06,2.708220e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572686,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
572687,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
572688,2022-12-31,True,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
572689,2022-12-31,False,2022,12,5,365,-0.974928,-0.222521,-2.449294e-16,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
